# 01 - Exploratory Data Analysis & Data Quality

**Goal:** understand the three source systems, find data quality issues, fix them, and link each finding to the user complaints from Task 1.

| Code | Complaint |
|---|---|
| C1 | Cannot find data / no overview |
| C2 | Data differs across systems |
| C3 | Salesforce data outdated/incorrect |
| C4 | Manual entry into multiple applications |

In [ ]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from src import config, quality, integrate
from src.load import load_all
from src.clean import clean_all
sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
FIG = config.FIGURES_DIR
def save(fig, name): fig.tight_layout(); fig.savefig(FIG / name, dpi=150)

## 1. What does each system contain?

In [ ]:
raw = load_all()
for s, df in raw.items():
    print(f"{s}: {df.shape[0]} rows x {df.shape[1]} columns ->", list(df.columns))

## 2. Raw data profile - missing values and placeholders
Users type `XX`, `N.A.`, `-`, `none`, `nan` instead of leaving a field empty. These hide missing data and turn numeric columns into text.

In [ ]:
profile = quality.profile_all(raw)
profile[profile.missing_pct > 0].sort_values("missing_pct", ascending=False)

## 3. Clean the data
All rules live in `src/clean.py`. Every rule writes how many rows it touched to a log.

In [ ]:
clean, log = clean_all(raw)
qlog = log.to_frame()
qlog[qlog.rows_affected > 0]

## 4. Key findings (with evidence)

### Finding 1 - Duplicate records (C2)

In [ ]:
for s in raw:
    ids = raw[s]["project_id"].replace("", np.nan).dropna()
    print(f"{s}: {ids.duplicated().sum()} duplicate IDs, exact duplicate rows: {raw[s].duplicated().sum()}")

All CT and MG duplicates are **exact copies** -> typical of a file being uploaded/imported twice (the manual Excel transfer). Fix: remove copies; in future enforce a unique key at load time.

### Finding 2 - Unit mix-up in ambient pressure (C4)

In [ ]:
p_raw = pd.to_numeric(raw["PPD"]["ambient_press"], errors="coerce")
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(p_raw.dropna(), bins=60, color="#c0504d"); axes[0].set_title("Before: two clusters (bar and mbar)")
axes[0].set_xlabel("ambient_press as entered")
axes[1].hist(clean["PPD"]["ambient_press"].dropna(), bins=40, color="#2a7f8e"); axes[1].set_title("After: all in bar")
axes[1].set_xlabel("ambient_press (bar)")
save(fig, "eda_pressure_units.png")
print((p_raw > 10).sum(), "values were entered in mbar instead of bar")

### Finding 3 - Missing project IDs and recovery (C1)

In [ ]:
print("PPD rows without ID:", (raw["PPD"]["project_id"] == "").sum())
qlog[qlog.column == "project_id"]

PPD and CT describe the same plant, so MW + plant type + region identify a project. An ID is restored only on a **unique** match. A simulation (blanking 470 known IDs) recovered 445 with **100 % correct** IDs and correctly refused the ambiguous ones.

### Finding 4 - 'Missing' costs that are not missing

In [ ]:
ct_raw = raw["CT"].replace(config.NULL_TOKENS, np.nan)
ct_raw.assign(gt_missing=ct_raw.cost_gt.isna(), st_missing=ct_raw.cost_st.isna()) \
      .groupby("plant_type")[["gt_missing", "st_missing"]].mean().round(2)

Steam plants (ST) have no gas turbine; single-cycle plants (SCGT) have no steam turbine. Those empty cells are **structural zeros**, not missing data. Blind imputation would invent a turbine cost for a plant that has no turbine.

### Finding 5 - Mixed date formats (C4)

In [ ]:
qlog[qlog.column == "handover_date"]

### Finding 6 - Fields without an owner: fuel type (C1)

In [ ]:
clean["PPD"].groupby("plant_type")["fuel_type"].value_counts().unstack(fill_value=0)

Fuel type is **never** filled for ST plants and only ~50 % for gas plants. The field has no clear owner or definition (what is the fuel of a steam plant?).

### Finding 7 - Values stuck at a maximum

In [ ]:
qlog[qlog.issue.str.contains("maximum")]

## 5. Data quality scorecard - before vs after

In [ ]:
score = quality.build_scorecard(raw, clean); score

## 6. Integration across systems
### Coverage - is every project in every system?

In [ ]:
cov = integrate.coverage_report(clean)
cov.systems.value_counts()  # how many projects exist in which systems

**~15 % of projects are missing from at least one system** -> no system has the full picture (C1).

### Consistency - do shared attributes agree?

In [ ]:
integrate.check_consistency(clean)

Once projects are correctly linked, shared attributes agree 100 %. So in this extract, "data differs" (C2) is caused by **duplicates, missing IDs, placeholders, unit and format differences** - i.e. by the manual transfer process - not by conflicting master data. (The Salesforce issue C3 cannot be tested directly: no raw Salesforce extract was provided.)

## 7. Characteristics of the data - what relates to winning?

In [ ]:
bid = integrate.build_bid_360(clean)
print("Overall win rate:", round(bid.win_loss.mean(), 3), "| projects:", len(bid))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col in zip(axes, ["plant_type", "region", "customer"]):
    r = bid.groupby(col).win_loss.mean().sort_values()
    r.plot.bar(ax=ax, color="#2a7f8e", rot=0); ax.axhline(bid.win_loss.mean(), ls="--", c="grey")
    ax.set_title(f"Win rate by {col}"); ax.set_ylim(0, 1); ax.set_xlabel("")
save(fig, "eda_winrate_segments.png")

## 8. Summary
- 7 data quality findings, each measured, fixed and mapped to a complaint.
- After cleaning: 100 % unique and linkable records in all systems.
- 1,699 of 2,000 projects exist in all three systems.
- Strongest visible win patterns: plant type (CCGT), region (EU), customer, efficiency.

Next: `02_model.ipynb`.